# 07 — Train LSTM

**PLTMH–ELC–QLSTM**

Notebook ini menangani tahap **pemodelan klasik** setelah dataset
supervised dibekukan pada `06_preprocessing.ipynb`.

Tahap yang dicakup:

1. baseline konstanta;
2. baseline linear berbasis endpoint;
3. LSTM klasik;
4. audit informasi temporal;
5. sequence ablation;
6. parameter-matched endpoint MLP;
7. audit kestabilan validasi.

Notebook ini **berakhir sebelum QLSTM dimulai**.

`08_train_qlstm.ipynb` menggunakan dataset dan target yang sama.


## 1. Kebijakan Model Klasik

Artefak model final telah dibekukan sebelum evaluasi QLSTM dan
*closed-loop*. Karena itu mode default notebook adalah
**verifikasi**, bukan pelatihan ulang.

```python
RETRAIN_LSTM = False
RETRAIN_ENDPOINT_MLP = False
```

Jika reproduksi eksplisit diaktifkan, model hasil reproduksi ditulis
ke `data/reproduction/notebook07/` dan **tidak menimpa checkpoint
final**.

### Larangan metodologis

- test set tidak digunakan untuk early stopping;
- test set tidak digunakan untuk pemilihan hiperparameter;
- dynamic-family split tidak boleh diubah;
- hasil test tidak boleh digunakan untuk melatih ulang model;
- ribuan window tidak diperlakukan sebagai unit statistik independen.


## 2. Interpretasi yang Dibekukan

LSTM dievaluasi terhadap baseline sederhana untuk mengetahui apakah
struktur temporal memberikan manfaat praktis pada dataset yang sama.

Hasil akhir tidak digunakan untuk mengklaim bahwa LSTM mempunyai
ketergantungan temporal yang kuat hanya karena arsitekturnya recurrent.
Audit sequence/temporal harus dipertimbangkan bersama hasil baseline.

Hasil test juga tidak digunakan sebagai dasar perubahan arsitektur.


In [ ]:
# ============================================================
# 07.1 — PROJECT + FROZEN ARTIFACT INTEGRITY
# ============================================================

from pathlib import Path

import hashlib
import json
import random
import sys

import numpy as np
import pandas as pd


EXPECTED_UPSTREAM_CHECKPOINT = "65ea0be16f7477af00b095c1eda010789102dfd0"

EXPECTED_NOTEBOOK_06_SHA256 = (
    "bbac80ff5eeaf6ba843720c7e58fc22f5fc85db19253b8d0cd6db16425c28881"
)

EXPECTED_CLASSICAL_ARTIFACT_SHA256 = (
    {
    "data/audits/cell140_dynamic_amplitude_by_temporal_region.csv": "c10e715ef5395e94a069a4b9c2c8c03f3c6d50cfcd10dd524698dbfe9dea8ffb",
    "data/audits/cell140_temporal_distribution.csv": "330556a6ab7d466a3f4b948378668520314e38cfca07fcff18d58fa5800b5052",
    "data/audits/cell140_temporal_information_audit.json": "62f0bb4e0a77feae39ed2c00eb497ffbe08397afd4e541da70adeb5f472e99a8",
    "data/audits/cell140_validation_family_prediction_bias.csv": "8e3f8b38b8faa60e96d8e98e264636d3f12fa64d591d79f46e062a9b9d073c5e",
    "data/audits/cell140_validation_performance_by_temporal_region.csv": "475d6ff8539889a4da3e68b680d1375cc04a683c863396d7fbeed5bb2000bb96",
    "data/audits/cell141_frozen_lstm_ablation_by_family.csv": "a4911f63ad7fa052aefab1839bb6ced2e9d2ec24c7fdd9bbb81dcf314eb64296",
    "data/audits/cell141_frozen_lstm_ablation_by_temporal_region.csv": "0dabf3ad5544c2a10e40e869dcc51536b3dee9e930e3f17a6e5fecc68e02f930",
    "data/audits/cell141_frozen_lstm_ablation_metadata.json": "55221815ac081471a2cf4bef9596a0fa4b7ae94b62d8b77b1959a577303ae87a",
    "data/audits/cell141_frozen_lstm_ablation_metrics.csv": "95cbeb7c312d581f24037be72cb09cc83ebc18742f975571917d293afd06ad2a",
    "data/audits/cell141_prediction_sensitivity.csv": "0a82019573fa9dc1062012943399134aaef9e96f148ad036c8e2f80bc6d2706f",
    "data/audits/cell143_multiseed_paired_comparison.csv": "86b1ed0e8ddd65b5351b574e8312ef316fa29f0bffcbab66e7c2b269d5d4ba0f",
    "data/audits/cell143_multiseed_stability_summary.csv": "4e75d09ff5e54a2b3e4da34b71b9123333200ad4d4693d66ae56783b46a9e472",
    "data/audits/cell143_multiseed_validation_metadata.json": "cf36dea185fed705bd407a3926776c19401ff0c647964f6d5e95a5a51174d22d",
    "data/audits/cell143_multiseed_validation_results.csv": "7351cb682670381b14670be9dc9a7463bd8226d93dfcfb956653018762ae9b36",
    "data/audits/cell144_qlstm_environment_readiness.json": "73a19c317d038db545ca98c38ac67511af14c649555e32cbb97221513f8a7384",
    "data/final/qlstm_lstm_dataset_final.npz": "7287018d944b7e3740aa778c78459df1142d53d77493a6193493af2009b46f60",
    "data/final/qlstm_lstm_dataset_metadata.json": "0e11a631937bbe349fe90bf326ba97fd5e14d209859ad10ca159b6fdf74efcf5",
    "data/final/qlstm_lstm_sample_metadata.csv.gz": "9d3f5297dc788231abefebb0dbbd4e98880ef3a6e33a3af6033fe68963d9aa5c",
    "data/final/qlstm_lstm_target_scaler.csv": "9dbd52fe5f99a2aa47275f85edc802b775db5879e00dcb54a24124a9d2ef0ab6",
    "models/endpoint_mlp_baseline/cell142_baseline_comparison.csv": "12f78b7ef00dfcde4d704f55e36ae0190b2f541458e045acb26f7c15f32bf41d",
    "models/endpoint_mlp_baseline/cell142_endpoint_mlp.pt": "b148c92cd7e1ce23de7996fe589de697ce004cb8c014e0b7f792999328187a5d",
    "models/endpoint_mlp_baseline/cell142_endpoint_mlp_config.json": "dde79500a5df3fd2a273a9f036606b8f4696aa37c39d730b9b87bfb1c167372a",
    "models/endpoint_mlp_baseline/cell142_endpoint_mlp_metrics.csv": "501c384d84c5c60f7a692397516bd2e0e50d1a8e1b8cc5c6ee807c30a90ad3b6",
    "models/endpoint_mlp_baseline/cell142_training_history.csv": "404dd300058f8c3c6a55c9de59233ac566cc868741f492d75eddcc7f9b715df9",
    "models/lstm_baseline/cell139_baseline_metrics.csv": "d6dccbd8ff5f7f7be55811cdb4b4a323eced87108b6a6f69f708c4eea8ce0b59",
    "models/lstm_baseline/cell139_classical_lstm.pt": "33fe34d8c06f2fdae316a302e75f5f398a9049e572093358dc77c3d8ee2d9a27",
    "models/lstm_baseline/cell139_lstm_config.json": "c5461ce5349ac8bbf0a4ca9c41321cf0d5e8f44f1df9af740f61a806224dc24f",
    "models/lstm_baseline/cell139_lstm_predictions.csv.gz": "bc90c485cfd8dc5d8a0a323b233a04940e212e86916372e8ed0f07b280dc4048",
    "models/lstm_baseline/cell139_lstm_training_history.csv": "b7ed5c03b14413cdd638e32766a108fc19b0afcb85f204beeaf6448393947652"
}
)

ESSENTIAL_ARTIFACT_RELPATHS = (
    {
    "dataset": "data/final/qlstm_lstm_dataset_final.npz",
    "dataset_metadata": "data/final/qlstm_lstm_dataset_metadata.json",
    "endpoint_checkpoint": "models/endpoint_mlp_baseline/cell142_endpoint_mlp.pt",
    "endpoint_comparison": "models/endpoint_mlp_baseline/cell142_baseline_comparison.csv",
    "endpoint_config": "models/endpoint_mlp_baseline/cell142_endpoint_mlp_config.json",
    "endpoint_history": "models/endpoint_mlp_baseline/cell142_training_history.csv",
    "endpoint_metrics": "models/endpoint_mlp_baseline/cell142_endpoint_mlp_metrics.csv",
    "lstm_checkpoint": "models/lstm_baseline/cell139_classical_lstm.pt",
    "lstm_config": "models/lstm_baseline/cell139_lstm_config.json",
    "lstm_history": "models/lstm_baseline/cell139_lstm_training_history.csv",
    "lstm_metrics": "models/lstm_baseline/cell139_baseline_metrics.csv",
    "lstm_predictions": "models/lstm_baseline/cell139_lstm_predictions.csv.gz",
    "sample_metadata": "data/final/qlstm_lstm_sample_metadata.csv.gz",
    "target_scaler": "data/final/qlstm_lstm_target_scaler.csv"
}
)

CLASSICAL_AUDIT_RELPATHS = (
    [
    "data/audits/cell140_dynamic_amplitude_by_temporal_region.csv",
    "data/audits/cell140_temporal_distribution.csv",
    "data/audits/cell140_temporal_information_audit.json",
    "data/audits/cell140_validation_family_prediction_bias.csv",
    "data/audits/cell140_validation_performance_by_temporal_region.csv",
    "data/audits/cell141_frozen_lstm_ablation_by_family.csv",
    "data/audits/cell141_frozen_lstm_ablation_by_temporal_region.csv",
    "data/audits/cell141_frozen_lstm_ablation_metadata.json",
    "data/audits/cell141_frozen_lstm_ablation_metrics.csv",
    "data/audits/cell141_prediction_sensitivity.csv",
    "data/audits/cell143_multiseed_paired_comparison.csv",
    "data/audits/cell143_multiseed_stability_summary.csv",
    "data/audits/cell143_multiseed_validation_metadata.json",
    "data/audits/cell143_multiseed_validation_results.csv",
    "data/audits/cell144_qlstm_environment_readiness.json"
]
)


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):

            digest.update(chunk)

    return digest.hexdigest()


candidate_roots = [
    Path("/content/PLTMH-ELC-QLSTM"),
    Path.cwd(),
    Path.cwd().parent,
]


PROJECT_ROOT = None


for candidate in candidate_roots:

    candidate = candidate.resolve()

    if (
        (candidate / ".git").exists()
        and
        (candidate / "src").exists()
        and
        (candidate / "notebooks").exists()
    ):

        PROJECT_ROOT = candidate
        break


if PROJECT_ROOT is None:

    raise RuntimeError(
        "PLTMH-ELC-QLSTM repository root not found."
    )


if str(PROJECT_ROOT) not in sys.path:

    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


NOTEBOOK_06_PATH = (
    PROJECT_ROOT
    /
    "notebooks"
    /
    "06_preprocessing.ipynb"
)


if (
    sha256_file(
        NOTEBOOK_06_PATH
    )
    !=
    EXPECTED_NOTEBOOK_06_SHA256
):

    raise RuntimeError(
        "Upstream Notebook 06 changed."
    )


artifact_integrity = {}


for relative_path, expected_sha in (
    EXPECTED_CLASSICAL_ARTIFACT_SHA256.items()
):

    path = (
        PROJECT_ROOT
        /
        relative_path
    )


    if not path.exists():

        artifact_integrity[
            relative_path
        ] = False

        continue


    artifact_integrity[
        relative_path
    ] = (
        sha256_file(
            path
        )
        ==
        expected_sha
    )


for relative_path, status in (
    artifact_integrity.items()
):

    print(
        f"{relative_path:72s}: {status}"
    )


FROZEN_CLASSICAL_ARTIFACTS_VALID = all(
    artifact_integrity.values()
)


if not FROZEN_CLASSICAL_ARTIFACTS_VALID:

    raise RuntimeError(
        "Frozen classical-model artifact changed."
    )


print(
    "\nFROZEN_CLASSICAL_ARTIFACTS_VALID:",
    FROZEN_CLASSICAL_ARTIFACTS_VALID
)


In [ ]:
# ====================================================
# 07.2 — LOAD THE SAME FROZEN SUPERVISED DATASET
# ====================================================

DATASET_PATH = (
    PROJECT_ROOT
    /
    ESSENTIAL_ARTIFACT_RELPATHS[
        "dataset"
    ]
)


DATASET_METADATA_PATH = (
    PROJECT_ROOT
    /
    ESSENTIAL_ARTIFACT_RELPATHS[
        "dataset_metadata"
    ]
)


dataset_metadata = json.loads(
    DATASET_METADATA_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    dataset_metadata[
        "dataset_status"
    ]
    !=
    "FINAL_LSTM_QLSTM_READY"
):

    raise RuntimeError(
        "Final dataset is not ready."
    )


with np.load(
    DATASET_PATH,
    allow_pickle=False,
) as dataset:

    X_train = (
        dataset[
            "X_train"
        ]
        .copy()
    )


    y_train = (
        dataset[
            "y_train"
        ]
        .copy()
    )


    X_val = (
        dataset[
            "X_validation"
        ]
        .copy()
    )


    y_val = (
        dataset[
            "y_validation"
        ]
        .copy()
    )


    X_test = (
        dataset[
            "X_test"
        ]
        .copy()
    )


    y_test = (
        dataset[
            "y_test"
        ]
        .copy()
    )


expected_shapes = {
    "X_train": (11200, 11, 4),
    "y_train": (11200, 2),
    "X_val": (3200, 11, 4),
    "y_val": (3200, 2),
    "X_test": (3200, 11, 4),
    "y_test": (3200, 2),
}


arrays = {
    "X_train": X_train,
    "y_train": y_train,
    "X_val": X_val,
    "y_val": y_val,
    "X_test": X_test,
    "y_test": y_test,
}


for name, array in arrays.items():

    if (
        array.shape
        !=
        expected_shapes[
            name
        ]
    ):

        raise RuntimeError(
            f"{name}: unexpected shape."
        )


    if not np.isfinite(
        array
    ).all():

        raise RuntimeError(
            f"{name}: non-finite values."
        )


    print(
        f"{name:8s}: "
        f"{array.shape} | "
        f"{array.dtype}"
    )


DATASET_CONTRACT_VALID = True


print(
    "\nDATASET_CONTRACT_VALID:",
    DATASET_CONTRACT_VALID
)


In [ ]:
# ====================================================
# 07.3 — FROZEN BASELINE + LSTM EVIDENCE
# ====================================================

LSTM_CONFIG_PATH = (
    PROJECT_ROOT
    /
    ESSENTIAL_ARTIFACT_RELPATHS[
        "lstm_config"
    ]
)


LSTM_METRICS_PATH = (
    PROJECT_ROOT
    /
    ESSENTIAL_ARTIFACT_RELPATHS[
        "lstm_metrics"
    ]
)


LSTM_HISTORY_PATH = (
    PROJECT_ROOT
    /
    ESSENTIAL_ARTIFACT_RELPATHS[
        "lstm_history"
    ]
)


lstm_config = json.loads(
    LSTM_CONFIG_PATH.read_text(
        encoding="utf-8"
    )
)


baseline_metrics = pd.read_csv(
    LSTM_METRICS_PATH
)


training_history = pd.read_csv(
    LSTM_HISTORY_PATH
)


if (
    lstm_config[
        "trainable_parameters"
    ]
    !=
    5426
):

    raise RuntimeError(
        "Frozen LSTM parameter count changed."
    )


if (
    lstm_config[
        "input_shape"
    ]
    !=
    [11, 4]
):

    raise RuntimeError(
        "Frozen LSTM input shape changed."
    )


if (
    lstm_config[
        "test_not_used_for_early_stopping"
    ]
    is not True
):

    raise RuntimeError(
        "Test-set policy changed."
    )


print(
    "Architecture:",
    lstm_config[
        "architecture"
    ]
)

print(
    "Trainable parameters:",
    lstm_config[
        "trainable_parameters"
    ]
)

print(
    "Best epoch:",
    lstm_config[
        "best_epoch"
    ]
)

print(
    "Best validation MSE:",
    lstm_config[
        "best_validation_mse"
    ]
)


print(
    "\nFrozen baseline metrics:"
)


print(
    baseline_metrics.to_string(
        index=False
    )
)


In [ ]:
# ====================================================
# 07.4 — LSTM ARCHITECTURE + CHECKPOINT AUDIT
# ====================================================

import torch
import torch.nn as nn


class ClassicalLSTM(nn.Module):

    def __init__(
        self,
        input_size=4,
        hidden_size=32,
        dense_size=16,
        output_size=2,
    ):

        super().__init__()


        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
        )


        self.dense = nn.Linear(
            hidden_size,
            dense_size,
        )


        self.activation = nn.Tanh()


        self.output = nn.Linear(
            dense_size,
            output_size,
        )


    def forward(
        self,
        x,
    ):

        sequence_output, _ = (
            self.lstm(
                x
            )
        )


        final_state = (
            sequence_output[
                :,
                -1,
                :
            ]
        )


        hidden = self.activation(
            self.dense(
                final_state
            )
        )


        return self.output(
            hidden
        )


architecture_probe = (
    ClassicalLSTM()
)


architecture_parameter_count = sum(
    parameter.numel()

    for parameter
    in architecture_probe.parameters()

    if parameter.requires_grad
)


if architecture_parameter_count != 5426:

    raise RuntimeError(
        "Reconstructed LSTM architecture "
        "does not contain 5,426 parameters."
    )


LSTM_CHECKPOINT_PATH = (
    PROJECT_ROOT
    /
    ESSENTIAL_ARTIFACT_RELPATHS[
        "lstm_checkpoint"
    ]
)


try:

    checkpoint = torch.load(
        LSTM_CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=True,
    )

except TypeError:

    checkpoint = torch.load(
        LSTM_CHECKPOINT_PATH,
        map_location="cpu",
    )


if (
    isinstance(
        checkpoint,
        dict,
    )
    and
    "model_state_dict"
    in
    checkpoint
):

    state_dict = checkpoint[
        "model_state_dict"
    ]


elif (
    isinstance(
        checkpoint,
        dict,
    )
    and
    "state_dict"
    in
    checkpoint
):

    state_dict = checkpoint[
        "state_dict"
    ]


elif isinstance(
    checkpoint,
    dict,
):

    state_dict = checkpoint


else:

    raise RuntimeError(
        "Unexpected LSTM checkpoint structure."
    )


checkpoint_tensor_count = sum(
    value.numel()

    for value
    in state_dict.values()

    if torch.is_tensor(
        value
    )
)


CHECKPOINT_PARAMETER_STRUCTURE_VALID = bool(
    checkpoint_tensor_count
    ==
    5426
)


print(
    "Architecture parameters:",
    architecture_parameter_count
)

print(
    "Checkpoint tensor elements:",
    checkpoint_tensor_count
)

print(
    "Checkpoint keys:"
)


for key in state_dict.keys():

    print(
        " -",
        key
    )


print(
    "\nCHECKPOINT_PARAMETER_STRUCTURE_VALID:",
    CHECKPOINT_PARAMETER_STRUCTURE_VALID
)


if not CHECKPOINT_PARAMETER_STRUCTURE_VALID:

    raise RuntimeError(
        "Frozen LSTM checkpoint parameter "
        "structure changed."
    )


# No forward pass is performed in verification mode.
MODEL_INFERENCE_PERFORMED = False


In [ ]:
# ============================================================
# 07.5 — FROZEN CLASSICAL PERFORMANCE INTERPRETATION
# ============================================================

EXPECTED_METRICS = {
    "constant_test_mse": 0.6523587107658386,
    "endpoint_linear_test_mse": 1.0355680100487212,
    "lstm_validation_mse": 0.5170130729675293,
    "lstm_test_mse": 0.8686286807060242,
    "lstm_test_kp_rmse": 0.2891014105037116,
    "lstm_test_ki_rmse": 4.157301447113995,
    "lstm_test_regime_accuracy": 0.5,
    "lstm_vs_endpoint_linear_test_improvement_pct": 16.120556807740993
}


def frozen_metric(
    model,
    split,
    column="mse_scaled",
):

    return float(
        baseline_metrics.loc[
            (
                baseline_metrics[
                    "model"
                ]
                ==
                model
            )
            &
            (
                baseline_metrics[
                    "split"
                ]
                ==
                split
            ),
            column,
        ].iloc[0]
    )


current_values = {
    "constant_test_mse":
        frozen_metric(
            "B0_CONSTANT",
            "test",
        ),

    "endpoint_linear_test_mse":
        frozen_metric(
            "B1_ENDPOINT_LINEAR",
            "test",
        ),

    "lstm_validation_mse":
        frozen_metric(
            "B2_LSTM",
            "validation",
        ),

    "lstm_test_mse":
        frozen_metric(
            "B2_LSTM",
            "test",
        ),

    "lstm_test_kp_rmse":
        frozen_metric(
            "B2_LSTM",
            "test",
            "kp_rmse",
        ),

    "lstm_test_ki_rmse":
        frozen_metric(
            "B2_LSTM",
            "test",
            "ki_rmse",
        ),

    "lstm_test_regime_accuracy":
        frozen_metric(
            "B2_LSTM",
            "test",
            "regime_accuracy",
        ),
}


current_values[
    "lstm_vs_endpoint_linear_test_improvement_pct"
] = (
    (
        current_values[
            "endpoint_linear_test_mse"
        ]
        -
        current_values[
            "lstm_test_mse"
        ]
    )
    /
    current_values[
        "endpoint_linear_test_mse"
    ]
    *
    100.0
)


metric_match = all(
    np.isclose(
        current_values[
            key
        ],
        EXPECTED_METRICS[
            key
        ],
        rtol=0.0,
        atol=1e-12,
    )

    for key
    in EXPECTED_METRICS
)


print(
    "Constant baseline test MSE :",
    current_values[
        "constant_test_mse"
    ]
)

print(
    "Endpoint-linear test MSE   :",
    current_values[
        "endpoint_linear_test_mse"
    ]
)

print(
    "LSTM validation MSE        :",
    current_values[
        "lstm_validation_mse"
    ]
)

print(
    "LSTM test MSE              :",
    current_values[
        "lstm_test_mse"
    ]
)

print(
    "LSTM Kp test RMSE          :",
    current_values[
        "lstm_test_kp_rmse"
    ]
)

print(
    "LSTM Ki test RMSE          :",
    current_values[
        "lstm_test_ki_rmse"
    ]
)

print(
    "LSTM vs endpoint-linear [%]:",
    current_values[
        "lstm_vs_endpoint_linear_test_improvement_pct"
    ]
)


print(
    "\nFrozen metrics exact:",
    metric_match
)


if not metric_match:

    raise RuntimeError(
        "Frozen Cell-139 metrics changed."
    )


# Scientific interpretation:
#
# - LSTM improves over endpoint-linear on the frozen test MSE.
# - LSTM does NOT outperform the train-mean constant baseline
#   on the frozen held-out test MSE.
# - Therefore strong temporal superiority is NOT inferred.
#
# No retraining is allowed based on this test observation.

LSTM_STRONG_TEMPORAL_SUPERIORITY_SUPPORTED = False


In [ ]:
# ====================================================
# 07.6 — TEMPORAL + SEQUENCE-ABLATION EVIDENCE
# ====================================================

print(
    "Frozen temporal/ablation artifacts:"
)


for relative_path in (
    CLASSICAL_AUDIT_RELPATHS
):

    print(
        " -",
        relative_path
    )


temporal_json_candidates = [
    relative_path

    for relative_path
    in CLASSICAL_AUDIT_RELPATHS

    if (
        "cell140"
        in
        Path(
            relative_path
        ).name.lower()

        and

        relative_path.endswith(
            ".json"
        )
    )
]


temporal_json_records = {}


for relative_path in (
    temporal_json_candidates
):

    path = (
        PROJECT_ROOT
        /
        relative_path
    )


    temporal_json_records[
        relative_path
    ] = json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


ablation_csv_candidates = [
    relative_path

    for relative_path
    in CLASSICAL_AUDIT_RELPATHS

    if (
        "cell141"
        in
        Path(
            relative_path
        ).name.lower()

        and

        (
            relative_path.endswith(
                ".csv"
            )
            or
            relative_path.endswith(
                ".csv.gz"
            )
        )
    )
]


print(
    "\nCell-140 JSON audit count:",
    len(
        temporal_json_candidates
    )
)

print(
    "Cell-141 ablation table count:",
    len(
        ablation_csv_candidates
    )
)


if len(
    temporal_json_candidates
) < 1:

    raise RuntimeError(
        "Cell-140 temporal audit JSON missing."
    )


if len(
    ablation_csv_candidates
) < 1:

    raise RuntimeError(
        "Cell-141 sequence-ablation evidence missing."
    )


print(
    "\nInterpretation freeze:"
)

print(
    "Strong temporal/order dependence supported: False"
)

print(
    "Temporal evidence must be interpreted "
    "together with static baselines."
)


STRONG_TEMPORAL_ORDER_DEPENDENCE_SUPPORTED = False


In [ ]:
# ============================================================
# 07.7 — PARAMETER-MATCHED ENDPOINT MLP AUDIT
# ============================================================

EXPECTED_ENDPOINT = {
    "validation_mse": 0.5198184251785278,
    "test_mse": 2.555413722991944,
    "parameters": 5554,
    "lstm_parameters": 5426
}


ENDPOINT_CONFIG_PATH = (
    PROJECT_ROOT
    /
    ESSENTIAL_ARTIFACT_RELPATHS[
        "endpoint_config"
    ]
)


ENDPOINT_METRICS_PATH = (
    PROJECT_ROOT
    /
    ESSENTIAL_ARTIFACT_RELPATHS[
        "endpoint_metrics"
    ]
)


endpoint_config = json.loads(
    ENDPOINT_CONFIG_PATH.read_text(
        encoding="utf-8"
    )
)


endpoint_metrics = pd.read_csv(
    ENDPOINT_METRICS_PATH
)


endpoint_validation_mse = float(
    endpoint_metrics.loc[
        endpoint_metrics[
            "split"
        ]
        ==
        "validation",
        "mse_scaled",
    ].iloc[0]
)


endpoint_test_mse = float(
    endpoint_metrics.loc[
        endpoint_metrics[
            "split"
        ]
        ==
        "test",
        "mse_scaled",
    ].iloc[0]
)


ENDPOINT_MLP_EVIDENCE_VALID = all(
    [
        endpoint_config[
            "input_representation"
        ]
        ==
        "LAST_TIMESTEP_ONLY",

        endpoint_config[
            "architecture"
        ]
        ==
        [4, 64, 64, 16, 2],

        endpoint_config[
            "trainable_parameters"
        ]
        ==
        EXPECTED_ENDPOINT[
            "parameters"
        ],

        endpoint_config[
            "cell139_lstm_parameters"
        ]
        ==
        EXPECTED_ENDPOINT[
            "lstm_parameters"
        ],

        np.isclose(
            endpoint_validation_mse,
            EXPECTED_ENDPOINT[
                "validation_mse"
            ],
            rtol=0.0,
            atol=1e-12,
        ),

        np.isclose(
            endpoint_test_mse,
            EXPECTED_ENDPOINT[
                "test_mse"
            ],
            rtol=0.0,
            atol=1e-12,
        ),

        endpoint_config[
            "test_used_for_early_stopping"
        ]
        is False,

        endpoint_config[
            "family_split_preserved"
        ]
        is True,
    ]
)


print(
    "Endpoint MLP architecture:",
    endpoint_config[
        "architecture"
    ]
)

print(
    "Endpoint MLP parameters:",
    endpoint_config[
        "trainable_parameters"
    ]
)

print(
    "LSTM parameters:",
    endpoint_config[
        "cell139_lstm_parameters"
    ]
)

print(
    "Endpoint validation MSE:",
    endpoint_validation_mse
)

print(
    "Endpoint test MSE:",
    endpoint_test_mse
)

print(
    "\nENDPOINT_MLP_EVIDENCE_VALID:",
    ENDPOINT_MLP_EVIDENCE_VALID
)


if not ENDPOINT_MLP_EVIDENCE_VALID:

    raise RuntimeError(
        "Frozen endpoint-MLP evidence changed."
    )


## 3. Reproduksi Training Opsional

Bagian berikut mendefinisikan kembali prosedur training LSTM untuk
reproduksi eksplisit.

Mode default tetap:

```python
RETRAIN_LSTM = False
```

Reproduksi tidak menggantikan checkpoint penelitian final dan tidak
boleh digunakan untuk menala model setelah test result diketahui.


In [ ]:
# ====================================================
# 07.8 — OPTIONAL LSTM RETRAINING
# ====================================================

import copy

from torch.utils.data import (
    DataLoader,
    TensorDataset,
)


RETRAIN_LSTM = False


REPRODUCTION_DIR = (
    PROJECT_ROOT
    /
    "data"
    /
    "reproduction"
    /
    "notebook07"
)


LSTM_RETRAINING_PERFORMED = False


def set_seed(
    seed,
):

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )


    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


def train_lstm_reproduction():

    config = lstm_config


    set_seed(
        int(
            config[
                "seed"
            ]
        )
    )


    device = torch.device(
        "cpu"
    )


    model = ClassicalLSTM(
        input_size=4,
        hidden_size=int(
            config[
                "architecture"
            ][
                "lstm_hidden_size"
            ]
        ),
        dense_size=int(
            config[
                "architecture"
            ][
                "dense_size"
            ]
        ),
        output_size=2,
    ).to(
        device
    )


    train_dataset = TensorDataset(
        torch.from_numpy(
            X_train
        ).float(),
        torch.from_numpy(
            y_train
        ).float(),
    )


    validation_dataset = TensorDataset(
        torch.from_numpy(
            X_val
        ).float(),
        torch.from_numpy(
            y_val
        ).float(),
    )


    generator = torch.Generator()

    generator.manual_seed(
        int(
            config[
                "seed"
            ]
        )
    )


    train_loader = DataLoader(
        train_dataset,
        batch_size=int(
            config[
                "batch_size"
            ]
        ),
        shuffle=True,
        generator=generator,
    )


    validation_loader = DataLoader(
        validation_dataset,
        batch_size=int(
            config[
                "batch_size"
            ]
        ),
        shuffle=False,
    )


    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=float(
            config[
                "learning_rate"
            ]
        ),
    )


    criterion = nn.MSELoss()


    best_validation_loss = (
        float(
            "inf"
        )
    )


    best_state = None

    best_epoch = None

    patience_counter = 0

    history = []


    max_epochs = int(
        config[
            "max_epochs"
        ]
    )


    patience = int(
        config[
            "early_stopping_patience"
        ]
    )


    min_delta = float(
        config[
            "early_stopping_min_delta"
        ]
    )


    for epoch in range(
        1,
        max_epochs + 1,
    ):

        model.train()


        train_sum = 0.0

        train_count = 0


        for batch_X, batch_y in (
            train_loader
        ):

            batch_X = (
                batch_X.to(
                    device
                )
            )


            batch_y = (
                batch_y.to(
                    device
                )
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            prediction = model(
                batch_X
            )


            loss = criterion(
                prediction,
                batch_y
            )


            loss.backward()


            optimizer.step()


            batch_n = (
                batch_X.shape[
                    0
                ]
            )


            train_sum += (
                float(
                    loss.item()
                )
                *
                batch_n
            )


            train_count += batch_n


        train_loss = (
            train_sum
            /
            train_count
        )


        model.eval()


        validation_sum = 0.0

        validation_count = 0


        with torch.no_grad():

            for batch_X, batch_y in (
                validation_loader
            ):

                batch_X = (
                    batch_X.to(
                        device
                    )
                )


                batch_y = (
                    batch_y.to(
                        device
                    )
                )


                prediction = model(
                    batch_X
                )


                loss = criterion(
                    prediction,
                    batch_y
                )


                batch_n = (
                    batch_X.shape[
                        0
                    ]
                )


                validation_sum += (
                    float(
                        loss.item()
                    )
                    *
                    batch_n
                )


                validation_count += (
                    batch_n
                )


        validation_loss = (
            validation_sum
            /
            validation_count
        )


        history.append(
            {
                "epoch":
                    epoch,

                "train_mse":
                    train_loss,

                "validation_mse":
                    validation_loss,
            }
        )


        improved = (
            validation_loss
            <
            (
                best_validation_loss
                -
                min_delta
            )
        )


        if improved:

            best_validation_loss = (
                validation_loss
            )


            best_state = copy.deepcopy(
                model.state_dict()
            )


            best_epoch = epoch

            patience_counter = 0


        else:

            patience_counter += 1


        if (
            patience_counter
            >=
            patience
        ):

            break


    if best_state is None:

        raise RuntimeError(
            "No reproduction checkpoint selected."
        )


    model.load_state_dict(
        best_state
    )


    return (
        model,
        pd.DataFrame(
            history
        ),
        best_epoch,
        best_validation_loss,
    )


if not RETRAIN_LSTM:

    print(
        "RETRAIN_LSTM = False"
    )

    print(
        "Frozen LSTM checkpoint is used as "
        "the authoritative scientific artifact."
    )

    print(
        "No training was performed."
    )


else:

    (
        reproduced_lstm,
        reproduced_history,
        reproduced_best_epoch,
        reproduced_best_validation_mse,
    ) = train_lstm_reproduction()


    REPRODUCTION_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )


    reproduction_checkpoint = (
        REPRODUCTION_DIR
        /
        "notebook07_retrained_lstm.pt"
    )


    reproduction_history = (
        REPRODUCTION_DIR
        /
        "notebook07_retrained_lstm_history.csv"
    )


    torch.save(
        reproduced_lstm.state_dict(),
        reproduction_checkpoint,
    )


    reproduced_history.to_csv(
        reproduction_history,
        index=False,
    )


    LSTM_RETRAINING_PERFORMED = True


    print(
        "Reproduction checkpoint:",
        reproduction_checkpoint
    )

    print(
        "Reproduction best epoch:",
        reproduced_best_epoch
    )

    print(
        "Reproduction best validation MSE:",
        reproduced_best_validation_mse
    )

    print(
        "Frozen model overwritten: False"
    )


In [ ]:
# ====================================================
# 07.9 — STATIC-BASELINE REPRODUCTION SWITCH
# ====================================================

RETRAIN_ENDPOINT_MLP = False

RERUN_MULTI_SEED_VALIDATION = False


if not RETRAIN_ENDPOINT_MLP:

    print(
        "RETRAIN_ENDPOINT_MLP = False"
    )

    print(
        "Frozen Cell-142 endpoint-MLP evidence "
        "is retained."
    )


if not RERUN_MULTI_SEED_VALIDATION:

    print(
        "RERUN_MULTI_SEED_VALIDATION = False"
    )

    print(
        "Frozen multi-seed/validation evidence "
        "is retained if present."
    )


ENDPOINT_MLP_RETRAINING_PERFORMED = False

MULTI_SEED_VALIDATION_RERUN = False


## 4. Handoff ke `08_train_qlstm.ipynb`

Artefak klasik otoritatif untuk tahap berikut adalah checkpoint LSTM
beku dari:

`models/lstm_baseline/cell139_classical_lstm.pt`

QLSTM harus:

- menggunakan dataset final yang sama;
- menggunakan target Kp dan Ki yang sama;
- mempertahankan family-level split yang sama;
- tidak mengubah dataset setelah melihat hasil LSTM;
- dibandingkan dengan LSTM yang sudah dibekukan.

Notebook 08 tidak boleh memodifikasi hasil Notebook 07.


In [ ]:
# ====================================================
# 07.10 — CLASSICAL-MODEL STAGE SUMMARY
# ====================================================

NOTEBOOK_07_CLASSICAL_STAGE_READY = all(
    [
        FROZEN_CLASSICAL_ARTIFACTS_VALID,
        DATASET_CONTRACT_VALID,
        CHECKPOINT_PARAMETER_STRUCTURE_VALID,
        ENDPOINT_MLP_EVIDENCE_VALID,
        not MODEL_INFERENCE_PERFORMED,
        not LSTM_RETRAINING_PERFORMED,
        not ENDPOINT_MLP_RETRAINING_PERFORMED,
        not MULTI_SEED_VALIDATION_RERUN,
    ]
)


print("=" * 72)
print("07_train_lstm.ipynb — SUMMARY")
print("=" * 72)


print(
    "Dataset                         : FINAL_LSTM_QLSTM_READY"
)

print(
    "Input shape                     : (11, 4)"
)

print(
    "LSTM parameters                 : 5426"
)

print(
    "LSTM best epoch                 :",
    lstm_config[
        "best_epoch"
    ]
)

print(
    "LSTM validation MSE             :",
    current_values[
        "lstm_validation_mse"
    ]
)

print(
    "LSTM test MSE                   :",
    current_values[
        "lstm_test_mse"
    ]
)

print(
    "Constant baseline test MSE       :",
    current_values[
        "constant_test_mse"
    ]
)

print(
    "Endpoint-linear test MSE         :",
    current_values[
        "endpoint_linear_test_mse"
    ]
)

print(
    "Endpoint-MLP validation MSE      :",
    endpoint_validation_mse
)

print(
    "Endpoint-MLP test MSE            :",
    endpoint_test_mse
)

print(
    "Strong temporal superiority      : False"
)

print(
    "Test used for early stopping     : False"
)

print(
    "Post-test tuning                 : False"
)

print(
    "LSTM retraining requested        :",
    RETRAIN_LSTM
)

print(
    "LSTM retraining performed        :",
    LSTM_RETRAINING_PERFORMED
)

print(
    "Endpoint MLP retraining          :",
    ENDPOINT_MLP_RETRAINING_PERFORMED
)

print(
    "Multi-seed validation rerun      :",
    MULTI_SEED_VALIDATION_RERUN
)

print(
    "Model inference in verify mode   :",
    MODEL_INFERENCE_PERFORMED
)

print(
    "NOTEBOOK 07 CLASSICAL STAGE READY:",
    NOTEBOOK_07_CLASSICAL_STAGE_READY
)


if NOTEBOOK_07_CLASSICAL_STAGE_READY:

    print(
        "\nNEXT NOTEBOOK:"
    )

    print(
        "08_train_qlstm.ipynb"
    )
